**Implementation of Variational Auto-encoder**

This hands‑on task follows an example‑driven approach: first understand the core VAE concepts by building and analyzing a VAE on the MNIST dataset, then apply the same methodology to the Fashion MNIST dataset to critically observe the differences in generative capabilities and latent space structure.

By the end of this lab, students will be able to:

Explain the key difference between a standard autoencoder and a VAE (deterministic vs. probabilistic latent space).

Understand the role of the reparameterization trick in enabling gradient‑based training.

Implement a VAE in TensorFlow/Keras using a custom train_step.

Visualize and interpret the continuous latent space of a VAE.

Generate new digits and smoothly interpolate between existing ones.

#### 🧑‍🎓 Student Details

| Field                | Information                |
|---------------------|----------------------------|
| 👨‍🎓 **Name**            | _[Fayad Mehamood]_     |
| 🆔 **USN**              | _[1RUA24CSE0141]_           |
| **Section**        | _[G]_ |
| **Programme**   | B.Tech(H)             |
| **School**  | Computer Science and Engineering |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set random seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)

In [ ]:
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

# Normalize pixel values to [0,1] and flatten
x_train = x_train.astype('float32') / 255.
x_test  = x_test.astype('float32') / 255.
x_train_flat = x_train.reshape((len(x_train), 784))
x_test_flat  = x_test.reshape((len(x_test), 784))

print("Training data shape:", x_train_flat.shape)
print("Test data shape:    ", x_test_flat.shape)

In [ ]:
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

# Normalize pixel values to [0,1] and flatten
x_train = x_train.astype('float32') / 255.
x_test  = x_test.astype('float32') / 255.
x_train_flat = x_train.reshape((len(x_train), 784))
x_test_flat  = x_test.reshape((len(x_test), 784))

print("Training data shape:", x_train_flat.shape)
print("Test data shape:    ", x_test_flat.shape)

In [ ]:
class Sampling(layers.Layer):
    """Samples z from a Gaussian distribution defined by z_mean and z_log_var."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim   = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
# --- Hyperparameters ---
input_dim  = 784
latent_dim = 2        # 2D for easy visualization

# --- Encoder ---
encoder_inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(256, activation='relu')(encoder_inputs)
x = layers.Dense(128, activation='relu')(x)
z_mean    = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])

encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')
encoder.summary()

In [ ]:
# --- Decoder ---
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(128, activation='relu')(latent_inputs)
x = layers.Dense(256, activation='relu')(x)
decoder_outputs = layers.Dense(input_dim, activation='sigmoid')(x)

decoder = keras.Model(latent_inputs, decoder_outputs, name='decoder')
decoder.summary()

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker      = keras.metrics.Mean(name="total_loss")
        self.recon_loss_tracker      = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker         = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        # data is (x, y) but we only need x (self-supervised)
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            reconstruction_loss = tf.reduce_mean(
                keras.losses.binary_crossentropy(data, reconstruction)
            ) * input_dim

            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )

            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.recon_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        if isinstance(data, tuple):
            data = data[0]
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)

        reconstruction_loss = tf.reduce_mean(
            keras.losses.binary_crossentropy(data, reconstruction)
        ) * input_dim

        kl_loss = -0.5 * tf.reduce_mean(
            1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
        )

        total_loss = reconstruction_loss + kl_loss

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.recon_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        return self.decoder(z)

In [ ]:
vae = VAE(encoder, decoder)
vae.compile(optimizer='adam')

In [ ]:
history = vae.fit(x_train_flat, x_train_flat,
                  epochs=30,
                  batch_size=128,
                  shuffle=True,
                  validation_data=(x_test_flat, x_test_flat),
                  verbose=1)

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Total Loss')
plt.legend()

plt.subplot(1,3,2)
plt.plot(history.history['reconstruction_loss'], label='Train')
plt.plot(history.history['val_reconstruction_loss'], label='Val')
plt.title('Reconstruction Loss')
plt.legend()

plt.subplot(1,3,3)
plt.plot(history.history['kl_loss'], label='Train')
plt.plot(history.history['val_kl_loss'], label='Val')
plt.title('KL Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def show_reconstructions(model, data, n=10):
    reconstructions = model.predict(data[:n], verbose=0)
    plt.figure(figsize=(12, 4))
    for i in range(n):
        plt.subplot(2, n, i+1)
        plt.imshow(data[i].reshape(28,28), cmap='gray')
        plt.axis('off')
        plt.subplot(2, n, i+n+1)
        plt.imshow(reconstructions[i].reshape(28,28), cmap='gray')
        plt.axis('off')
    plt.show()

show_reconstructions(vae, x_test_flat)

**Final Task for Students**

Objective
Apply the exact same VAE architecture and training procedure to the Fashion MNIST dataset. Compare the generative quality with the MNIST results.


**Students Observations on :**

Question 01. Reconstruction Quality: How does the reconstruction quality of Fashion MNIST compare to MNIST for the same latent dimension (2)? Is the difference more pronounced than with a standard autoencoder? Explain.

Question 02. Generative Quality: Observe the grid of generated Fashion MNIST images. Are the generated items coherent? What common failure modes do you observe (e.g., blurry, missing parts)?


All the best !!
Ashwini Kumar Mathur